In [6]:
import asyncio # Imports the library for running asynchronous code.
from agents import Agent, Runner, set_tracing_disabled # Imports the core components from the OpenAI Agents SDK.
import os 
# Disable the SDK's tracing feature to keep the output clean for this tutorial.
set_tracing_disabled(True)

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
# Define a simple agent for a quick test.
agent = Agent(
    name="Assistant", # Assign a name to the agent.
    instructions="Reply very concisely.", # Provide a simple instruction to guide its behavior.
    # Specify the model using the LiteLLM provider format for Nebius.
    model="litellm/nebius/moonshotai/Kimi-K2-Instruct",
)

# Execute the agent with a test prompt. 'await' is used because the run is an asynchronous operation.
result = await Runner.run(
    agent, # The agent to run.
    "Tell me why it is important to evaluate AI agents." # The user's input prompt.
)

# Print the final text output from the agent's response.
print(result.final_output)

Evaluation ensures AI agents are safe, reliable, and aligned with human goals by identifying failures, biases, and misuse risks before deployment.


In [7]:
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class MemoryNote:
    text: str
    last_update_date: str
    keywords: List[str]


In [10]:
@dataclass
class TravelState:
    profile: Dict[str, Any] = field(default_factory=dict)

    global_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    session_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    trip_history: Dict[str, Any] = field(default_factory=lambda: {"trips":[]})

    system_frontmatter: str = ""
    global_memories_md: str = ""
    session_memories_md:str = ""

    inject_session_memories_next_turn: bool = False

In [13]:
user_state = TravelState(
    profile={
        "global_customer_id": "crm_12345",
        "name": "John Doe",
        "age": "31",
        "home_city": "San Francisco",
        "currency" : "USD",
        "passport_expiry_date": "2029-06-12",
        "loyalty_status": {"airline": "United Gold", "hotel": "Marriott Titanium"},
        "loyalty_ids": {"marriott": "MR998877", "hilton": "HH445566", "hyatt": "HY112233"},
        "seat_preference": "aisle",
        "tone": "concise and friendly",
        "active_visas": ["Schengen", "US"],
        "insurance_coverage_profile": {
            "car_rental": "primary_cdw_included",
            "travel_medical": "covered",
        },
    },
    global_memory = {
        "notes": [
            # Each note is an instance of MemoryNote, converted to a dictionary for JSON compatibility.
            MemoryNote(
                text="For trips shorter than a week, user generally prefers not to check bags.",
                last_update_date="2025-04-05",
                keywords=["baggage", "short_trip"],
            ).__dict__,
            MemoryNote(
                text="User usually prefers aisle seats.",
                last_update_date="2024-06-25",
                keywords=["seat_preference"],
            ).__dict__,
            MemoryNote(
                text="User generally likes central, walkable city-center neighborhoods.",
                last_update_date="2024-02-11",
                keywords=["neighborhood"],
            ).__dict__,
            MemoryNote(
                text="User generally likes to compare options side-by-side",
                last_update_date="2023-02-17",
                keywords=["pricing"],
            ).__dict__,
            MemoryNote(
                text="User prefers high floors",
                last_update_date="2023-02-11",
                keywords=["room"],
            ).__dict__,
        ]
    },
    trip_history = {
        "trips": [
            {
                # Core trip details
                "from_city": "Istanbul",
                "from_country": "Turkey",
                "to_city": "Paris",
                "to_country": "France",
                "check_in_date": "2025-05-01",
                "check_out_date": "2025-05-03",
                "trip_purpose": "leisure",  # leisure | business | family | etc.
                "party_size": 1,

                # Flight details
                "flight": {
                    "airline": "United",
                    "airline_status_at_booking": "United Gold",
                    "cabin_class": "economy_plus",
                    "seat_selected": "aisle",
                    "seat_location": "front",          # front | middle | back
                    "layovers": 1,
                    "baggage": {"checked_bags": 0, "carry_ons": 1},
                    "special_requests": ["vegetarian_meal"],  # optional
                },

                # Hotel details
                "hotel": {
                    "brand": "Hilton",
                    "property_name": "Hilton Paris Opera",
                    "neighborhood": "city_center",
                    "bed_type": "king",
                    "smoking": "non_smoking",
                    "high_floor": True,
                    "early_check_in": False,
                    "late_check_out": True,
                },
            }
        ]
    },
)

In [ ]:
from datetime import datetime,timezone
from typing import List
from agents import function_tool,RunContextWrapper


def _today_iso_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT")


@function_tool
def save_memory_note(ctx: RunContextWrapper[TravelState],text:str, keywords = List[str]) -> dict:
    if "notes" not in ctx.context.session_memory or ctx.context.session_memory["notes"] is None:
        ctx.context.session_memory["notes"] = []

    clean_keywords = [
        k.strip().lower()
        for k in keywords if isinstance(k,str) and k.strip()
        ][:3]

    ctx.context.session_memory["notes"].append({
        "text": text.strip(),
        "last_update_date": _today_iso_utc(),
        "keywords": clean_keywords
    })
    return {"ok":True}
